In [ ]:
import time

# 1. Defining the GRU Model Architecture
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, hn = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

gru_model = GRUModel(input_dim=1, hidden_dim=32, layer_dim=2, output_dim=1).to(device)
optimizer_gru = optim.Adam(gru_model.parameters(), lr=0.001)

# 2. Training the GRU Model (Training Loop) and Time Measurement
print("GRU Model Training Begins...")
start_time = time.time()
gru_model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        outputs = gru_model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer_gru.zero_grad()
        loss.backward()
        optimizer_gru.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - GRU Eğitim Kaybı (Loss): {total_loss/len(train_loader):.4f}")

gru_training_time = time.time() - start_time
print(f"GRU Eğitim Süresi: {gru_training_time:.2f} saniye")

# 3. GRU Estimates
gru_model.eval()
with torch.no_grad():
    gru_predictions = gru_model(X_test_device)

# 4. Performance Comparison (MSE Metric)
mse_criterion = nn.MSELoss()
lstm_loss = mse_criterion(predictions, y_test_tensor.to(device)).item()
gru_loss = mse_criterion(gru_predictions, y_test_tensor.to(device)).item()

print("\n--- MODEL COMPARISON REPORT ---")
print(f"LSTM Test MSE (Hata) Skoru: {lstm_loss:.6f}")
print(f"GRU Test MSE (Hata) Skoru:  {gru_loss:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

# 1. Defining the GRU Model Architecture
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, hn = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

gru_model = GRUModel(input_dim=1, hidden_dim=32, layer_dim=2, output_dim=1).to(device)
optimizer_gru = optim.Adam(gru_model.parameters(), lr=0.001)

# 2. Training the GRU Model (Training Loop) and Time Measurement
print("GRU Model Training Begins...")
start_time = time.time()
gru_model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        outputs = gru_model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer_gru.zero_grad()
        loss.backward()
        optimizer_gru.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - GRU Eğitim Kaybı (Loss): {total_loss/len(train_loader):.4f}")

gru_training_time = time.time() - start_time
print(f"GRU Eğitim Süresi: {gru_training_time:.2f} saniye")

# 3. GRU Estimates
gru_model.eval()
with torch.no_grad():
    gru_predictions = gru_model(X_test_device)

# 4. Performance Comparison (MSE Metric)
mse_criterion = nn.MSELoss()
lstm_loss = mse_criterion(predictions, y_test_tensor.to(device)).item()
gru_loss = mse_criterion(gru_predictions, y_test_tensor.to(device)).item()

print("\n--- MODEL COMPARISON REPORT ---")
print(f"LSTM Test MSE (Hata) Skoru: {lstm_loss:.6f}")
print(f"GRU Test MSE (Hata) Skoru:  {gru_loss:.6f}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import time

# 1. Device Identification
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan Hesaplama Cihazı: {device}")

# 2. Data Loading and Preprocessing

df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
solar_data = df[['AT_solar_generation_actual']].dropna()

scaler = MinMaxScaler(feature_range=(-1, 1))
solar_data_scaled = scaler.fit_transform(solar_data.values)

# 3. Sliding Window
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

lookback = 24 
X, y = create_sequences(solar_data_scaled, lookback)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# 4. Tensor Transformations and DataLoader
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float()

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
X_test_device = X_test_tensor.to(device)
y_test_device = y_test_tensor.to(device)

criterion = nn.MSELoss()
epochs = 5

# =============================================
# 5. LSTM MODEL TRAINING AND PREDICTION

# =============================================
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

lstm_model = LSTMModel(input_dim=1, hidden_dim=32, layer_dim=2, output_dim=1).to(device)
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

print("\n--- LSTM Model Training Begins ---")
lstm_model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer_lstm.zero_grad()
        loss.backward()
        optimizer_lstm.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - LSTM Loss: {total_loss/len(train_loader):.4f}")

lstm_model.eval()
with torch.no_grad():
    lstm_predictions = lstm_model(X_test_device)

# =============================================
# 6. GRU MODEL TRAINING AND PREDICTION
# =============================================
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, hn = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

gru_model = GRUModel(input_dim=1, hidden_dim=32, layer_dim=2, output_dim=1).to(device)
optimizer_gru = optim.Adam(gru_model.parameters(), lr=0.001)

print("\n--- GRU Model Training Begins ---")
gru_model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = gru_model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer_gru.zero_grad()
        loss.backward()
        optimizer_gru.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - GRU Loss: {total_loss/len(train_loader):.4f}")

gru_model.eval()
with torch.no_grad():
    gru_predictions = gru_model(X_test_device)

# =============================================
# 7. PERFORMANCE COMPARISON REPORT
# =============================================
lstm_loss = criterion(lstm_predictions, y_test_device).item()
gru_loss = criterion(gru_predictions, y_test_device).item()

print("\n==========================================")
print("MODEL COMPARISON REPORT")
print("===========================================")
print(f"LSTM Test MSE (Hata) Skoru: {lstm_loss:.6f}")
print(f"GRU Test MSE (Hata) Skoru:  {gru_loss:.6f}")
print("===========================================")